# P053 — Day 1 Production Training on Colab A100
## Memory Yield Predictor — HybridTransformerCNN

**This notebook trains the Day 1 champion model ONLY.**

All subsequent retraining (Day 31 drift, Day 39 rollback) happens
**autonomously on AWS EC2** via Airflow DAGs — not in this notebook.

**Architecture:**
```
NB03 (Colab A100) → Day 1 champion → Google Drive → S3
                                                    ↓
AWS EC2 (Airflow) → 40-day simulation → drift detection → auto-retrain → canary → promote/rollback
```

**What this notebook does:**
1. Generate 16M-row DRAM dataset (10M train + 2M val + 2M test + 2M unseen)
2. Train HybridTransformerCNN (317K params), 50 epochs, bfloat16 AMP
3. Evaluate on val/test/unseen splits
4. Save model weights + benchmark to Google Drive
5. Upload champion model to S3 (for AWS consumption)

**Key:** After this notebook, you never open Colab again. Everything runs on AWS.

In [ ]:
# ━━━ Installation (Colab only) ━━━
import subprocess, sys

packages = [
    "mlflow>=2.15", "torch", "scikit-learn", "numpy", "pandas",
    "matplotlib", "psutil",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\u2705 All packages installed")

In [ ]:
# ━━━ Google Drive Mount & Project Setup ━━━
import os
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone or update repo
REPO_URL = "https://github.com/AIML-Engineering-Lab/053_dram_memory_yield_mlops.git"
PROJECT_DIR = Path("/content/053_memory_yield_predictor")

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    os.system("git pull origin main")
    print("✅ Repository updated")
else:
    os.system(f"git clone {REPO_URL} {PROJECT_DIR}")
    os.chdir(PROJECT_DIR)
    print("✅ Repository cloned")

# Add src to path
sys.path.insert(0, str(PROJECT_DIR))
print(f" Working directory: {os.getcwd()}")

In [ ]:
# ━━━ Hardware Detection ━━━
import torch
import psutil

def detect_hardware():
    """Auto-detect GPU and return optimal training settings."""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        cc = torch.cuda.get_device_capability(0)

        if cc[0] >= 8:  # A100/H100/L4 (cc >= 8.0)
            amp_dtype = "bfloat16"
            use_amp = True
            use_gradscaler = False
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        else:  # T4/V100 (cc < 8.0)
            amp_dtype = "float16"
            use_amp = True
            use_gradscaler = True

        device = torch.device("cuda")
    else:
        gpu_name = "CPU"
        vram_gb = 0
        amp_dtype = "float32"
        use_amp = False
        use_gradscaler = False
        device = torch.device("cpu")

    return {
        "device": device, "gpu_name": gpu_name, "vram_gb": round(vram_gb, 1),
        "amp_dtype": amp_dtype, "use_amp": use_amp, "use_gradscaler": use_gradscaler,
    }

hw = detect_hardware()
print("=" * 60)
print(f"  GPU:         {hw['gpu_name']}")
print(f"  VRAM:        {hw['vram_gb']} GB")
print(f"  AMP dtype:   {hw['amp_dtype']}")
print(f"  GradScaler:  {hw['use_gradscaler']}")
print(f"  RAM:         {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"  CPU cores:   {psutil.cpu_count()}")
print("=" * 60)

if hw['gpu_name'] == 'CPU':
    print("⚠️  WARNING: No GPU detected! Training will be extremely slow.")
    print("    Go to Runtime → Change runtime type → GPU (A100 preferred)")

In [ ]:
# ━━━ Generate Full Dataset (16M rows) ━━━
import time

DRIVE_DATA = Path("/content/drive/MyDrive/P053_artifacts/data")
LOCAL_DATA = PROJECT_DIR / "data"
LOCAL_DATA.mkdir(exist_ok=True)

preprocessed_path = LOCAL_DATA / "preprocessed_full.npz"

# Check if preprocessed data exists on Drive
if (DRIVE_DATA / "preprocessed_full.npz").exists():
    import shutil
    shutil.copy(DRIVE_DATA / "preprocessed_full.npz", preprocessed_path)
    print(f"\u2705 Loaded preprocessed data from Drive ({preprocessed_path.stat().st_size / 1e9:.1f} GB)")
elif preprocessed_path.exists():
    print(f"\u2705 Preprocessed data already exists locally ({preprocessed_path.stat().st_size / 1e9:.1f} GB)")
else:
    print("\u23f3 Generating full 16M-row dataset...")
    t0 = time.time()
    from src.data_generator import generate_all_splits
    generate_all_splits()
    print(f"  Data generation: {time.time() - t0:.1f}s")

    print("\u23f3 Preprocessing...")
    t0 = time.time()
    from src.preprocess import preprocess_and_save
    preprocess_and_save(full=True)
    print(f"  Preprocessing: {time.time() - t0:.1f}s")

    # Save to Drive for persistence
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy(preprocessed_path, DRIVE_DATA / "preprocessed_full.npz")
    print(f"\u2705 Data saved to Google Drive for future runs")

# Verify
import numpy as np
data = np.load(preprocessed_path, allow_pickle=True)
print(f"\n Dataset Shape:")
print(f"  Train:  {data['X_train'].shape} ({data['y_train'].sum():.0f} fails)")
print(f"  Val:    {data['X_val'].shape}")
print(f"  Test:   {data['X_test'].shape}")
print(f"  Unseen: {data['X_unseen'].shape}")
print(f"  Features: {len(data['feature_names'])} \u2192 {list(data['feature_names'][:5])}...")

In [ ]:
# ━━━ MLflow Setup (local tracking on Colab instance) ━━━
import mlflow

MLFLOW_DIR = PROJECT_DIR / "mlruns_colab"
MLFLOW_DIR.mkdir(exist_ok=True)

# Set local tracking for Colab (no network dependency)
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{PROJECT_DIR / 'mlflow_colab.db'}"

from src.mlflow_utils import init_mlflow
experiment_id = init_mlflow("P053-Production-Training")
print(f"\u2705 MLflow initialized")
print(f"  Experiment ID: {experiment_id}")
print(f"  Tracking URI: {mlflow.get_tracking_uri()}")

## Session 1: Day 1 — Initial Production Training

Full training on 16M-row DRAM dataset. 50 epochs with early stopping (patience=12).
This establishes the champion model baseline.

In [ ]:
# ━━━ SESSION 1: Day 1 — Initial Training (50 epochs) ━━━
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score,
)

from src.config import MODEL_PARAMS, TRAINING
from src.model import HybridTransformerCNN, create_dataloaders, find_best_threshold
from src.focal_loss import FocalLossWithLabelSmoothing
from src.mlflow_utils import (
    start_training_run, log_epoch_metrics, log_evaluation_results,
    log_model_artifact, log_training_summary,
)

# \u2500\u2500\u2500 Training helper \u2500\u2500\u2500
def train_one_epoch(model, loader, criterion, optimizer, device, scaler, hw_info):
    model.train()
    total_loss = 0
    n_batches = 0
    sample_preds, sample_labels = [], []

    amp_ctx = (
        torch.autocast("cuda", dtype=torch.bfloat16 if hw_info["amp_dtype"] == "bfloat16" else torch.float16)
        if hw_info["use_amp"]
        else torch.autocast("cpu", enabled=False)
    )

    for batch_idx, batch in enumerate(loader):
        x_tab, x_spa, labels = [b.to(device) for b in batch]
        optimizer.zero_grad(set_to_none=True)

        with amp_ctx:
            logits = model(x_tab, x_spa)
            loss = criterion(logits, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        if batch_idx % 10 == 0:
            with torch.no_grad():
                preds = torch.sigmoid(logits).float().cpu().numpy()
                sample_preds.extend(preds)
                sample_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / n_batches
    auc_pr = average_precision_score(np.array(sample_labels), np.array(sample_preds)) if np.array(sample_labels).sum() > 0 else 0
    return avg_loss, auc_pr


@torch.no_grad()
def evaluate_split(model, X, y, feature_names, device, criterion, hw_info, batch_size=2048):
    model.eval()
    spatial_cols = ["die_x", "die_y", "edge_distance"]
    spatial_idx = [feature_names.index(c) for c in spatial_cols if c in feature_names]
    tabular_idx = [i for i in range(len(feature_names)) if i not in spatial_idx]

    X_tab = torch.tensor(X[:, tabular_idx].astype(np.float32))
    X_spa = torch.tensor(X[:, spatial_idx].astype(np.float32))

    amp_ctx = (
        torch.autocast("cuda", dtype=torch.bfloat16 if hw_info["amp_dtype"] == "bfloat16" else torch.float16)
        if hw_info["use_amp"]
        else torch.autocast("cpu", enabled=False)
    )

    all_logits = []
    total_loss = 0
    n_batches = 0

    for i in range(0, len(X_tab), batch_size):
        xt = X_tab[i:i+batch_size].to(device)
        xs = X_spa[i:i+batch_size].to(device)
        lb = torch.tensor(y[i:i+batch_size].astype(np.float32)).to(device)
        with amp_ctx:
            logits = model(xt, xs)
            loss = criterion(logits, lb)
        all_logits.append(logits.cpu())
        total_loss += loss.item()
        n_batches += 1

    logits = torch.cat(all_logits)
    proba = torch.sigmoid(logits).float().numpy()
    avg_loss = total_loss / n_batches
    auc_pr = average_precision_score(y, proba) if y.sum() > 0 else 0
    return avg_loss, auc_pr, proba


def run_training_session(
    session_name, run_name, data, hw_info,
    epochs=50, lr=1e-3, batch_size=4096, patience=12,
    model_weights_path=None
):
    """Complete training session with MLflow tracking.

    Args:
        session_name: Description (e.g., "Day 1 Initial Training")
        run_name: MLflow run name
        data: dict with X_train, y_train, X_val, y_val, X_test, y_test, X_unseen, y_unseen, feature_names
        hw_info: hardware detection result
        epochs: max epochs
        lr: learning rate
        batch_size: batch size
        patience: early stopping patience
        model_weights_path: path to pretrained weights (for fine-tuning)
    """
    device = hw_info["device"]
    t_global = time.time()

    print(f"\n{'='*70}")
    print(f"  {session_name}")
    print(f"  Run: {run_name} | GPU: {hw_info['gpu_name']} | AMP: {hw_info['amp_dtype']}")
    print(f"  LR: {lr} | Batch: {batch_size} | Epochs: {epochs} | Patience: {patience}")
    print(f"{'='*70}\n")

    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val = data["X_val"], data["y_val"]
    X_test, y_test = data["X_test"], data["y_test"]
    X_unseen, y_unseen = data["X_unseen"], data["y_unseen"]
    feature_names = list(data["feature_names"])

    # Dataloaders
    train_loader, val_loader, n_tab, n_spa = create_dataloaders(
        X_train, y_train, X_val, y_val, feature_names,
        batch_size=batch_size, oversample=True,
    )

    # Model
    model = HybridTransformerCNN(
        n_tabular=n_tab, n_spatial=n_spa,
        d_model=MODEL_PARAMS["d_model"], n_heads=MODEL_PARAMS["n_heads"],
        n_layers=MODEL_PARAMS["n_layers"], cnn_out=MODEL_PARAMS["cnn_out"],
        dropout=MODEL_PARAMS["dropout"],
    ).to(device)

    if model_weights_path and Path(model_weights_path).exists():
        model.load_state_dict(torch.load(model_weights_path, weights_only=True, map_location=device))
        print(f"  \u2705 Loaded pretrained weights from {model_weights_path}")

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {n_params:,}")

    # Loss, Optimizer, Scheduler
    criterion = FocalLossWithLabelSmoothing(
        alpha=TRAINING["focal_alpha"], gamma=TRAINING["focal_gamma"],
        smoothing=TRAINING["label_smoothing"],
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=TRAINING["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if hw_info["use_gradscaler"] else None

    # MLflow run
    run_ctx = start_training_run(
        run_name=run_name, gpu_name=hw_info["gpu_name"], amp_dtype=hw_info["amp_dtype"],
        batch_size=batch_size, learning_rate=lr,
        extra_params={
            "hw.vram_gb": hw_info["vram_gb"],
            "hw.use_gradscaler": str(hw_info["use_gradscaler"]),
            "train.rows": int(len(y_train)),
            "session": session_name,
            "pretrained": str(model_weights_path is not None),
        },
        extra_tags={
            "gpu_type": hw_info["gpu_name"],
            "run_context": "colab",
            "session": session_name,
        },
    )

    with run_ctx as run:
        print(f"  MLflow run: {run.info.run_id}\n")

        history = {"train_loss": [], "val_loss": [], "train_auc_pr": [], "val_auc_pr": []}
        epoch_times = []
        best_val_auc = 0
        best_epoch = 0
        patience_counter = 0
        save_path = PROJECT_DIR / "src" / "artifacts" / f"hybrid_best_{run_name}.pt"
        save_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"{'Ep':>4} {'TrLoss':>8} {'VaLoss':>8} {'TrAUC':>8} {'VaAUC':>8} {'LR':>10} {'Time':>7}")
        print("-" * 70)

        for epoch in range(1, epochs + 1):
            t_epoch = time.time()
            train_loss, train_auc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, hw_info)
            val_loss, val_auc, _ = evaluate_split(model, X_val, y_val, feature_names, device, criterion, hw_info)

            scheduler.step()
            epoch_time = time.time() - t_epoch
            epoch_times.append(epoch_time)
            throughput = len(y_train) / epoch_time

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_auc_pr"].append(train_auc)
            history["val_auc_pr"].append(val_auc)

            current_lr = optimizer.param_groups[0]["lr"]
            log_epoch_metrics(epoch, train_loss, val_loss, train_auc, val_auc, current_lr, epoch_time, throughput)

            improved = ""
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_epoch = epoch
                patience_counter = 0
                improved = " \u2605"
                torch.save(model.state_dict(), save_path)
            else:
                patience_counter += 1

            if epoch <= 5 or epoch % 5 == 0 or improved:
                print(f"{epoch:>4} {train_loss:>8.4f} {val_loss:>8.4f} {train_auc:>8.4f} {val_auc:>8.4f} {current_lr:>10.6f} {epoch_time:>6.1f}s{improved}")

            if patience_counter >= patience:
                print(f"\n[STOP]  Early stopping at epoch {epoch} (best: {best_epoch})")
                break

        # Load best weights
        model.load_state_dict(torch.load(save_path, weights_only=True, map_location=device))

        # Evaluate all splits
        print(f"\n{'='*70}")
        print(f"  EVALUATION \u2014 best model from epoch {best_epoch}")
        print(f"{'='*70}")

        results = {}
        threshold = None
        for split_name, X_split, y_split in [("val", X_val, y_val), ("test", X_test, y_test), ("unseen", X_unseen, y_unseen)]:
            _, split_auc, proba = evaluate_split(model, X_split, y_split, feature_names, device, criterion, hw_info)
            if split_name == "val":
                threshold = find_best_threshold(y_split, proba)
            y_pred = (proba >= threshold).astype(int)
            cm = confusion_matrix(y_split, y_pred)
            tn, fp, fn, tp = cm.ravel()
            results[split_name] = {
                "precision": float(precision_score(y_split, y_pred, zero_division=0)),
                "recall": float(recall_score(y_split, y_pred, zero_division=0)),
                "f1": float(f1_score(y_split, y_pred, zero_division=0)),
                "auc_pr": float(split_auc),
                "auc_roc": float(roc_auc_score(y_split, proba)),
                "threshold": float(threshold),
                "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
            }
            print(f"  {split_name.upper():>7}: AUC-PR={split_auc:.4f} | F1={results[split_name]['f1']:.4f} | Recall={results[split_name]['recall']:.4f} | TP={tp} FP={fp} FN={fn} TN={tn}")

        # MLflow logging
        log_evaluation_results(results, threshold)

        total_time = time.time() - t_global
        peak_vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

        log_training_summary(
            best_epoch=best_epoch, best_val_auc=best_val_auc,
            total_time_min=total_time / 60, avg_epoch_time_s=np.mean(epoch_times),
            throughput=len(y_train) / min(epoch_times) if epoch_times else 0,
            peak_vram_gb=peak_vram, epochs_run=len(epoch_times), train_rows=len(y_train),
        )

        if save_path.exists():
            log_model_artifact(save_path)

        # Save benchmark
        import json
        benchmark = {
            "session": session_name, "run_name": run_name,
            "device": str(device), "gpu_name": hw_info["gpu_name"],
            "gpu_vram_gb": hw_info["vram_gb"], "amp_dtype": hw_info["amp_dtype"],
            "batch_size": batch_size, "lr": lr, "model_params": n_params,
            "train_rows": len(y_train), "epochs_run": len(epoch_times),
            "best_epoch": best_epoch, "best_val_auc": best_val_auc,
            "avg_epoch_time_s": float(np.mean(epoch_times)),
            "total_train_time_min": float(total_time / 60),
            "peak_gpu_memory_gb": float(peak_vram),
            "results": results, "mlflow_run_id": run.info.run_id,
        }

        benchmark_path = PROJECT_DIR / "data" / f"benchmark_{run_name}.json"
        with open(benchmark_path, "w") as f:
            json.dump(benchmark, f, indent=2)
        mlflow.log_artifact(str(benchmark_path), "benchmark")

        print(f"\n[DONE] {session_name} complete in {total_time/60:.1f} min")
        print(f"   Best Val AUC-PR: {best_val_auc:.4f} (epoch {best_epoch})")
        print(f"   Peak VRAM: {peak_vram:.1f} GB")
        print(f"   MLflow run: {run.info.run_id}")

    return {
        "run_id": run.info.run_id,
        "best_epoch": best_epoch,
        "best_val_auc": best_val_auc,
        "results": results,
        "save_path": str(save_path),
        "benchmark": benchmark,
    }

print("\u2705 Training functions defined")

In [ ]:
# ━━━ Run Session 1: Day 1 Initial Training ━━━
session1 = run_training_session(
    session_name="Day 1 \u2014 Initial Production Training",
    run_name="A100-day1-initial",
    data=data,
    hw_info=hw,
    epochs=50,
    lr=1e-3,
    batch_size=4096,
    patience=12,
)

print(f"\n Session 1 Summary:")
for split, metrics in session1["results"].items():
    print(f"  {split}: AUC-PR={metrics['auc_pr']:.4f} | F1={metrics['f1']:.4f}")

## Save Day 1 Champion to Google Drive

Model weights, benchmark JSON, and MLflow DB are saved to Drive for persistence.
These will be downloaded to your Mac and uploaded to S3 for the AWS 40-day simulation.

In [ ]:
# ━━━ Save Day 1 Champion to Google Drive ━━━
import shutil, json

DRIVE_ARTIFACTS = Path("/content/drive/MyDrive/P053_artifacts")
DRIVE_MODELS = DRIVE_ARTIFACTS / "models"
DRIVE_BENCHMARKS = DRIVE_ARTIFACTS / "benchmarks"
DRIVE_MLFLOW = DRIVE_ARTIFACTS / "mlflow"

for d in [DRIVE_MODELS, DRIVE_BENCHMARKS, DRIVE_MLFLOW]:
    d.mkdir(parents=True, exist_ok=True)

# Save Day 1 model weights
src = Path(session1["save_path"])
if src.exists():
    dst = DRIVE_MODELS / src.name
    shutil.copy(src, dst)
    print(f"✅ Day 1 model → {dst} ({src.stat().st_size / 1e6:.1f} MB)")

# Save benchmark
dst = DRIVE_BENCHMARKS / f"benchmark_{session1['benchmark']['run_name']}.json"
with open(dst, "w") as f:
    json.dump(session1["benchmark"], f, indent=2)
print(f"✅ Benchmark → {dst}")

# Save MLflow database
mlflow_db = PROJECT_DIR / "mlflow_colab.db"
if mlflow_db.exists():
    shutil.copy(mlflow_db, DRIVE_MLFLOW / "mlflow_colab.db")
    print(f"✅ MLflow DB → {DRIVE_MLFLOW / 'mlflow_colab.db'}")

# Save comparison plot if generated
plot_path = PROJECT_DIR / "assets" / "nb03_training_curves.png"
if plot_path.exists():
    shutil.copy(plot_path, DRIVE_ARTIFACTS / "nb03_training_curves.png")

print(f"\n All Day 1 artifacts saved to: {DRIVE_ARTIFACTS}")
print(f"   Next: Download to Mac → Upload model to S3 → Start AWS 40-day simulation")

## Summary & Next Steps

### Day 1 Results
| Metric | Val | Test | Unseen |
|--------|-----|------|--------|
| AUC-PR | TBD | TBD | TBD |
| F1 | TBD | TBD | TBD |
| Recall | TBD | TBD | TBD |

### Artifacts on Drive
- **Model weights:** `/content/drive/MyDrive/P053_artifacts/models/hybrid_best_A100-day1-initial.pt`
- **Benchmark:** `/content/drive/MyDrive/P053_artifacts/benchmarks/benchmark_A100-day1-initial.json`
- **MLflow DB:** `/content/drive/MyDrive/P053_artifacts/mlflow/mlflow_colab.db`

### What Happens Next (ALL on AWS — never come back to Colab)

```
1. Download artifacts from Drive to your Mac
2. Upload Day 1 model to S3: aws s3 cp model.pt s3://p053-mlflow-artifacts/models/
3. Launch EC2 + RDS (Steps 7.3-8 in AWS_SETUP_GUIDE.md)
4. Deploy Docker stack on EC2 (MLflow + FastAPI + Airflow + Kafka + Spark)
5. Trigger dag_simulation_master → runs 40 days automatically:
   - Daily: 2-9M rows → Kafka → Spark ETL → inference → drift check
   - Day 31: Auto-retrain ON AWS (15-day sliding window)
   - Day 39: Bad deploy → canary catches → rollback
6. Monitor via Grafana dashboards + MLflow UI
```

**This notebook is DONE. Close Colab. Everything from here runs on AWS.**

In [ ]:
# ━━━ Final Summary ━━━
print("=" * 70)
print("  P053 — DAY 1 CHAMPION MODEL TRAINED")
print("=" * 70)
print(f"  GPU:               {hw['gpu_name']}")
print(f"  VRAM:              {hw['vram_gb']} GB")
print(f"  AMP:               {hw['amp_dtype']}")
print(f"  Training rows:     {session1['benchmark']['train_rows']:,}")
print(f"  Epochs run:        {session1['benchmark']['epochs_run']}")
print(f"  Best epoch:        {session1['best_epoch']}")
print(f"  Val AUC-PR:        {session1['best_val_auc']:.4f}")
print(f"  Test AUC-PR:       {session1['results']['test']['auc_pr']:.4f}")
print(f"  Unseen AUC-PR:     {session1['results']['unseen']['auc_pr']:.4f}")
print(f"  Test F1:           {session1['results']['test']['f1']:.4f}")
print(f"  Training time:     {session1['benchmark']['total_train_time_min']:.1f} min")
print(f"  Peak VRAM:         {session1['benchmark']['peak_gpu_memory_gb']:.1f} GB")
print(f"  MLflow run:        {session1['run_id']}")
print(f"  Model saved:       {session1['save_path']}")
print("=" * 70)
print()
print("  ✅ Day 1 complete. Close Colab.")
print("  ➡️  Next: AWS EC2 deployment + 40-day simulation via Airflow")
print("=" * 70)